In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

daily_demand = pd.read_csv("../data/processed/daily_demand.csv")
daily_demand['order_date'] = pd.to_datetime(daily_demand['order_date'])
daily_demand = daily_demand.sort_values('order_date').reset_index(drop=True)

print(daily_demand.shape)
daily_demand.tail()

Importing plotly failed. Interactive plots will not work.


(524, 3)


,order_date,total_qty,total_revenue
519,2026-06-25,32,22346.0
520,2026-06-26,104,66701.5
521,2026-06-27,122,112028.0
522,2026-06-28,64,42458.5
523,2026-06-29,175,84988.5


In [2]:
test_size = 30
train = daily_demand.iloc[:-test_size].copy()
test = daily_demand.iloc[-test_size:].copy()

print("Train:", train.shape, "| Test:", test.shape)
print("Train ends:", train['order_date'].max())
print("Test range:", test['order_date'].min(), "to", test['order_date'].max())

Train: (494, 3) | Test: (30, 3)
Train ends: 2026-05-30 00:00:00
Test range: 2026-05-31 00:00:00 to 2026-06-29 00:00:00


In [3]:
results = []

def evaluate(name, actual, predicted):
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = mean_absolute_percentage_error(actual, predicted) * 100
    results.append({"model": name, "RMSE": round(rmse, 2), "MAPE (%)": round(mape, 2)})
    print(f"{name} -> RMSE: {rmse:.2f}, MAPE: {mape:.2f}%")

In [4]:
def make_features(df):
    df = df.copy()
    df['day_num'] = (df['order_date'] - daily_demand['order_date'].min()).dt.days
    df['day_of_week'] = df['order_date'].dt.dayofweek
    df['month'] = df['order_date'].dt.month
    return df

train_feat = make_features(train)
test_feat = make_features(test)

feature_cols = ['day_num', 'day_of_week', 'month']

lr = LinearRegression()
lr.fit(train_feat[feature_cols], train_feat['total_qty'])
lr_pred = lr.predict(test_feat[feature_cols])

evaluate("Linear Regression", test_feat['total_qty'], lr_pred)

Linear Regression -> RMSE: 41.00, MAPE: 32.89%


In [5]:
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(train_feat[feature_cols], train_feat['total_qty'])
rf_pred = rf.predict(test_feat[feature_cols])

evaluate("Random Forest", test_feat['total_qty'], rf_pred)

Random Forest -> RMSE: 45.70, MAPE: 32.77%


In [6]:
xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42)
xgb.fit(train_feat[feature_cols], train_feat['total_qty'])
xgb_pred = xgb.predict(test_feat[feature_cols])

evaluate("XGBoost", test_feat['total_qty'], xgb_pred)

XGBoost -> RMSE: 51.03, MAPE: 39.22%


In [7]:
arima_model = ARIMA(train['total_qty'], order=(5,1,2))
arima_fit = arima_model.fit()
arima_pred = arima_fit.forecast(steps=test_size)

evaluate("ARIMA", test['total_qty'].values, arima_pred.values)

ARIMA -> RMSE: 41.60, MAPE: 31.55%


In [8]:
sarima_model = SARIMAX(train['total_qty'], order=(1,1,1), seasonal_order=(1,1,1,7))
sarima_fit = sarima_model.fit(disp=False)
sarima_pred = sarima_fit.forecast(steps=test_size)

evaluate("SARIMA", test['total_qty'].values, sarima_pred.values)

SARIMA -> RMSE: 40.66, MAPE: 32.96%


In [9]:
prophet_train = train[['order_date','total_qty']].rename(columns={'order_date':'ds','total_qty':'y'})

prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True)
prophet_model.fit(prophet_train)

future = prophet_model.make_future_dataframe(periods=test_size)
prophet_forecast = prophet_model.predict(future)
prophet_pred = prophet_forecast.tail(test_size)['yhat'].values

evaluate("Prophet", test['total_qty'].values, prophet_pred)

14:06:25 - cmdstanpy - INFO - Chain [1] start processing
14:06:26 - cmdstanpy - INFO - Chain [1] done processing


Prophet -> RMSE: 41.41, MAPE: 39.10%


In [10]:
results_df = pd.DataFrame(results).sort_values('RMSE')
print(results_df)

results_df.to_csv("../reports/model_comparison.csv", index=False)
print("\nBest model:", results_df.iloc[0]['model'])

               model   RMSE  MAPE (%)
4             SARIMA  40.66     32.96
0  Linear Regression  41.00     32.89
5            Prophet  41.41     39.10
3              ARIMA  41.60     31.55
1      Random Forest  45.70     32.77
2            XGBoost  51.03     39.22

Best model: SARIMA


In [11]:
final_sarima = SARIMAX(daily_demand['total_qty'], order=(1,1,1), seasonal_order=(1,1,1,7))
final_sarima_fit = final_sarima.fit(disp=False)

forecast_result = final_sarima_fit.get_forecast(steps=30)
future_dates = pd.date_range(start=daily_demand['order_date'].max() + pd.Timedelta(days=1), periods=30)

sarima_forecast_df = pd.DataFrame({
    'date': future_dates,
    'predicted_qty': forecast_result.predicted_mean.values,
    'lower_bound': forecast_result.conf_int()['lower total_qty'].values,
    'upper_bound': forecast_result.conf_int()['upper total_qty'].values
})

sarima_forecast_df.to_csv("../reports/demand_forecast.csv", index=False)
print("Saved new SARIMA-based demand_forecast.csv")
sarima_forecast_df.head()

Saved new SARIMA-based demand_forecast.csv


,date,predicted_qty,lower_bound,upper_bound
0,2026-06-30,121.381286,57.807251,184.955321
1,2026-07-01,121.835807,58.050386,185.621227
2,2026-07-02,117.607313,53.660296,181.554330
3,2026-07-03,113.063419,48.955759,177.171080
4,2026-07-04,121.241592,56.973696,185.509488
